# DEMO: XỬ LÝ DỮ LIỆU CDC TỪ KAFKA (ORACLE OLTP) SANG BẢNG VALUE VÀ BẢNG HISTORY

Notebook này mô phỏng luồng xử lý dữ liệu CDC (Change Data Capture) từ nguồn Oracle OLTP qua Kafka Topic (Bronze CDC):
1. **Bảng Value (`customer_value`)**: Ánh xạ snapshot 1:1 với trạng thái hiện tại của OLTP (SCD Type 1 - MERGE INTO với deduplication).
2. **Bảng History (`customer_history`)**: Lưu vết toàn bộ lịch sử biến động của các thao tác Insert, Update, Delete theo thời gian (Audit Trail).
3. **Xử lý Chống lặp dữ liệu (Idempotency)**: Giải pháp MERGE INTO cho Bảng History khi chạy lại (replay) mốc CDC cũ.

## BƯỚC 1: KHỞI TẠO VÀ CẤU HÌNH SPARK & DELTA LAKE

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, current_timestamp, lit, when
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

# Khởi tạo Spark Session với cấu hình Delta Lake
spark = SparkSession.builder \
    .appName("CDC_Oracle_Kafka_Merge_Value_And_History") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("Spark Session & Delta Lake đã sẵn sàng!")

## BƯỚC 2: MÔ PHỎNG DỮ LIỆU BAN ĐẦU (VERSION 0) CHO BẢNG VALUE & HISTORY

In [ ]:
# 1. Bảng Value (Silver Current) - Lưu trạng thái snapshot hiện tại 1:1 với Oracle OLTP
initial_value_data = [
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 15000000.0, datetime(2026, 1, 1, 8, 0)),
    ('CUST_102', 'Tran Thi Binh', 'Danang', 20000000.0, datetime(2026, 1, 1, 8, 0)),
    ('CUST_103', 'Le Van Cuong', 'HCM', 10000000.0, datetime(2026, 1, 1, 8, 0))
]
val_schema = ['customer_id', 'customer_name', 'address', 'credit_limit', 'updated_at']
df_val_init = spark.createDataFrame(initial_value_data, val_schema)
df_val_init.write.format('delta').mode('overwrite').saveAsTable('customer_value')

# 2. Bảng History (Silver History) - Lưu lịch sử biến động
initial_history_data = [
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 15000000.0, datetime(2026, 1, 1, 8, 0), 'INSERT', datetime(2026, 1, 1, 8, 0)),
    ('CUST_102', 'Tran Thi Binh', 'Danang', 20000000.0, datetime(2026, 1, 1, 8, 0), 'INSERT', datetime(2026, 1, 1, 8, 0)),
    ('CUST_103', 'Le Van Cuong', 'HCM', 10000000.0, datetime(2026, 1, 1, 8, 0), 'INSERT', datetime(2026, 1, 1, 8, 0))
]
hist_schema = ['customer_id', 'customer_name', 'address', 'credit_limit', 'updated_at', 'cdc_op', 'ingest_at']
df_hist_init = spark.createDataFrame(initial_history_data, hist_schema)
df_hist_init.write.format('delta').mode('overwrite').saveAsTable('customer_history')

print('--- BẢNG VALUE BAN ĐẦU ---')
spark.table('customer_value').show()

## BƯỚC 3: MÔ PHỎNG LUỒNG BRONZE CDC TỪ KAFKA (ORACLE CDC LOGS)

In [ ]:
# Mô phỏng Bronze CDC Data chứa thông tin log từ Oracle CDC (GoldenGate / Debezium)
# cdc_op: 'I' (Insert), 'U' (Update), 'D' (Delete)
bronze_cdc_data = [
    # CUST_101: Update hạn mức tín dụng lúc 09:00, sau đó Update địa chỉ lúc 10:00
    ('CUST_101', 'Nguyen Van An', 'Hanoi', 18000000.0, datetime(2026, 1, 1, 9, 0), 'U'),
    ('CUST_101', 'Nguyen Van An', 'Haiphong', 18000000.0, datetime(2026, 1, 1, 10, 0), 'U'),
    
    # CUST_102: Khách hàng bị xóa khỏi hệ thống OLTP lúc 10:15
    ('CUST_102', 'Tran Thi Binh', 'Danang', 20000000.0, datetime(2026, 1, 1, 10, 15), 'D'),
    
    # CUST_104: Khách hàng mới đăng ký lúc 10:30
    ('CUST_104', 'Pham Dang Khoa', 'Cantho', 25000000.0, datetime(2026, 1, 1, 10, 30), 'I')
]

bronze_schema = ['customer_id', 'customer_name', 'address', 'credit_limit', 'cdc_timestamp', 'cdc_op']
df_bronze_cdc = spark.createDataFrame(bronze_cdc_data, bronze_schema)

print('--- DỮ LIỆU BRONZE CDC RAW TỪ KAFKA ---')
df_bronze_cdc.show()

## BƯỚC 4: KHỬ TRÙNG DỮ LIỆU CDC (DEDUPLICATION IN BATCH)

Lọc ra log có `cdc_timestamp` mới nhất cho từng `customer_id` trong Batch trước khi MERGE vào Bảng Value.

In [ ]:
# Dùng Window Function lấy bản ghi CDC mới nhất theo cdc_timestamp cho mỗi customer_id
window_spec = Window.partitionBy('customer_id').orderBy(col('cdc_timestamp').desc())

df_cdc_latest = df_bronze_cdc \
    .withColumn('rn', row_number().over(window_spec)) \
    .filter(col('rn') == 1) \
    .drop('rn')

print('--- DỮ LIỆU CDC ĐÃ DEDUPLICATE ---')
df_cdc_latest.show()

## BƯỚC 5: MERGE VÀO BẢNG VALUE (CURRENT SNAPSHOT 1:1 VỚI NGUỒN OLTP)

In [ ]:
# Kết nối với Bảng Delta customer_value
tbl_value = DeltaTable.forName(spark, 'customer_value')

# Thực hiện MERGE INTO:
tbl_value.alias('tgt').merge(
    df_cdc_latest.alias('src'),
    'tgt.customer_id = src.customer_id'
).whenMatchedDelete(
    condition = "src.cdc_op = 'D'"
).whenMatchedUpdate(
    condition = "src.cdc_op = 'U'",
    set = {
        'customer_name': 'src.customer_name',
        'address': 'src.address',
        'credit_limit': 'src.credit_limit',
        'updated_at': 'src.cdc_timestamp'
    }
).whenNotMatchedInsert(
    condition = "src.cdc_op = 'I'",
    values = {
        'customer_id': 'src.customer_id',
        'customer_name': 'src.customer_name',
        'address': 'src.address',
        'credit_limit': 'src.credit_limit',
        'updated_at': 'src.cdc_timestamp'
    }
).execute()

print('--- BẢNG VALUE SAU KHI MERGE ---')
spark.table('customer_value').show()

## BƯỚC 6: NẠP DỮ LIỆU VÀO BẢNG HISTORY (CHỐNG LẶP DỮ LIỆU BẰNG MERGE - IDEMPOTENT)

In [ ]:
# Thay vì dùng .mode('append') có nguy cơ bị lặp khi replay mốc CDC cũ,
# ta dùng MERGE INTO dựa trên Event Composite Key (customer_id + cdc_timestamp + cdc_op)

df_history_source = df_bronze_cdc \
    .withColumnRenamed('cdc_timestamp', 'updated_at') \
    .withColumn('ingest_at', current_timestamp())

tbl_history = DeltaTable.forName(spark, 'customer_history')

# Chỉ INSERT sự kiện CDC nếu nó CHƯA XUẤT HIỆN trong bảng History (Chống trùng khi replay)
tbl_history.alias('tgt').merge(
    df_history_source.alias('src'),
    'tgt.customer_id = src.customer_id AND tgt.updated_at = src.updated_at AND tgt.cdc_op = src.cdc_op'
).whenNotMatchedInsert(
    values = {
        'customer_id': 'src.customer_id',
        'customer_name': 'src.customer_name',
        'address': 'src.address',
        'credit_limit': 'src.credit_limit',
        'updated_at': 'src.updated_at',
        'cdc_op': 'src.cdc_op',
        'ingest_at': 'src.ingest_at'
    }
).execute()

print('--- BẢNG HISTORY SAU KHI NẠP DỮ LIỆU (IDEMPOTENT MERGE) ---')
spark.table('customer_history').orderBy('customer_id', 'updated_at').show()

## BƯỚC 7: THỬ NHIỆM THỰC TẾ: REPLAY LẠI MỐC CDC CŨ XEM CÓ BỊ TRÙNG LẶP KHÔNG

In [ ]:
# Chạy lại chính xác df_history_source lần thứ 2 (Mô phỏng replay Kafka offset)
print('--- TIẾN HÀNH CHẠY LẠI (REPLAY) ĐỢT CDC CŨ ---')
tbl_history.alias('tgt').merge(
    df_history_source.alias('src'),
    'tgt.customer_id = src.customer_id AND tgt.updated_at = src.updated_at AND tgt.cdc_op = src.cdc_op'
).whenNotMatchedInsert(
    values = {
        'customer_id': 'src.customer_id',
        'customer_name': 'src.customer_name',
        'address': 'src.address',
        'credit_limit': 'src.credit_limit',
        'updated_at': 'src.updated_at',
        'cdc_op': 'src.cdc_op',
        'ingest_at': 'src.ingest_at'
    }
).execute()

print('--- KẾT QUẢ BẢNG HISTORY VẪN DỮ NGUYÊN SỐ LƯỢNG DÒNG (KHÔNG BỊ TRÙNG LẶP) ---')
spark.table('customer_history').orderBy('customer_id', 'updated_at').show()